In [1]:
# Notebook A – Geometry & Decomposition (MoGe + Grounded-SAM)
# =====================================================================
# This notebook produces:
#   - scene point cloud (scene_cloud.ply)
#   - per‑object masks (masks/obj_XX.png)
#   - per‑object 3D clusters (clusters/obj_XX.ply)
#   - clean object crops (objects/obj_XX.png)
#   - floor plane (floor.ply)
#   - background room mesh (background_room.glb)
#   - meta.json with intrinsics and object metadata
#
# After initial inference, you can upload Gemini‑edited images
# to synchronise the crops, masks, and clusters.
# =====================================================================

In [1]:
# A0 – Runtime check
# ---------------------------------------------------------------------
import subprocess, sys
print("Python version:", sys.version.split()[0])
gpu_info = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print("GPU:", gpu_info or "NO GPU – enable T4")

Python version: 3.12.13
GPU: Tesla T4, 15360 MiB


In [ ]:
# ---------------------------------------------------------------------
# A1 – Install dependencies (final robust version)
# ---------------------------------------------------------------------
import sys, subprocess, os

# 1. Uninstall any numpy (to avoid version conflicts)
!pip uninstall -y numpy

# 2. Install numpy 1.24.4 (compatible with open3d 0.19.0)
!pip install --no-cache-dir "numpy==1.24.4"

# 3. Install core packages with pinned versions (use --no-deps to avoid pulling newer numpy)
!pip install --no-cache-dir "open3d==0.19.0" "xatlas==0.0.9" "trimesh==4.4.9" \
  "plotly==5.24.1" "opencv-python-headless==4.10.0.84" "Pillow==10.4.0" \
  "scipy==1.13.1" "torch" "torchvision" --no-deps
# Now install them normally (to get their true dependencies, but numpy is already fixed)
!pip install --no-cache-dir "open3d==0.19.0" "xatlas==0.0.9" "trimesh==4.4.9" \
  "plotly==5.24.1" "opencv-python-headless==4.10.0.84" "Pillow==10.4.0" \
  "scipy==1.13.1"

# 4. Install transformers (avoid upgrading numpy)
!pip install --no-cache-dir "transformers>=4.44" "sentencepiece" "protobuf" "accelerate" --no-deps
!pip install --no-cache-dir "transformers>=4.44" "sentencepiece" "protobuf" "accelerate"

# 5. Re-install numpy 1.24.4 (in case any package downgraded/upgraded it)
!pip install --no-cache-dir --force-reinstall --no-deps "numpy==1.24.4"

# 6. Clone MoGe and install it as a package WITHOUT build isolation
if not os.path.exists("/content/MoGe"):
    !git clone -q https://github.com/microsoft/MoGe.git /content/MoGe

# Install dependencies needed by MoGe
!pip install --no-cache-dir einops kornia

# Install MoGe in editable mode, skipping its dependency resolution
!pip install -e /content/MoGe --no-build-isolation --no-deps

# Ensure the MoGe package is available in sys.path (just in case)
MOGE_PATH = "/content/MoGe"
if MOGE_PATH not in sys.path:
    sys.path.insert(0, MOGE_PATH)

# 7. Verify imports
print("\n=== Verification ===")
import importlib
for m in ["open3d", "trimesh", "transformers", "cv2", "scipy", "numpy", "torch", "einops", "kornia"]:
    try:
        ver = getattr(importlib.import_module(m), "__version__", "?")
        print(f" ✅ {m} {ver}")
    except Exception as e:
        print(f" ❌ {m} – {e}")

import numpy as np
print(f"NumPy version: {np.__version__}")
if not np.__version__.startswith("1."):
    print(" ⚠️  WARNING: NumPy 2.x detected – open3d may fail")
else:
    print(" ✅ NumPy version is compatible")

# Test MoGe import (try different possible import paths)
moges = ["moge.model.v2", "moge.model.v1", "moge.model", "moge"]
found = False
for modpath in moges:
    try:
        mod = importlib.import_module(modpath)
        print(f" ✅ MoGe imported from {modpath}")
        found = True
        break
    except ImportError:
        continue
if not found:
    print(" ❌ MoGe import failed – try adding the repo path to PYTHONPATH manually:")
    print("    import sys; sys.path.insert(0, '/content/MoGe')")
else:
    print(" ✅ MoGe is ready.")

print("\nIf open3d still fails, restart runtime and run this cell again.")

In [1]:
# A2 – Config, imports, helpers
# ---------------------------------------------------------------------
import os, time, json, io, re, glob, shutil, gc
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import open3d as o3d
import cv2

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)
for sub in ["", "/clusters", "/objects", "/meshes", "/masks"]:
    os.makedirs(OUT + sub, exist_ok=True)

CFG = {
    "OBJECTS": [
        "sofa with pillows and blanket",
        "coffee table with books, mug and candle",
        "large potted plant with pot and stand",
        "small potted plant with pot",
        "floor lamp",
        "area rug",
        "basket with magazines"
    ],
    "BOX_THR": 0.30,
    "TXT_THR": 0.25,
    "MOGE_ID": "Ruicheng/moge-2-vitl-normal",
    "MAX_DIM": 1024,
    "REL_JUMP": 0.05,
    "VIZ_PTS": 60000,
}

_t = time.time()
def log(m): print(f"[{time.time()-_t:6.1f}s] {m}", flush=True)
def banner(m): print("\n" + "="*70 + f"\n  {m}\n" + "="*70, flush=True)

def show_pcd(P, C, title, mx=CFG["VIZ_PTS"]):
    if len(P) > mx:
        i = np.random.choice(len(P), mx, replace=False)
        P, C = P[i], C[i]
    c = ["rgb(%d,%d,%d)" % tuple((np.clip(C,0,1)*255).astype(int)[i]) for i in range(len(P))]
    fig = go.Figure(go.Scatter3d(x=P[:,0], y=P[:,1], z=P[:,2],
                                 mode="markers", marker=dict(size=1.3, color=c)))
    fig.update_layout(title=title, height=540,
                      margin=dict(l=0,r=0,t=30,b=0),
                      scene=dict(aspectmode="data"))
    fig.show()

def show_mesh(m, title, mx=200000):
    v = np.asarray(m.vertices)
    f = np.asarray(m.triangles)
    vc = np.asarray(m.vertex_colors) if m.has_vertex_colors() else None
    if len(f) > mx:
        f = f[np.random.choice(len(f), mx, replace=False)]
    kw = dict(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2])
    if vc is not None and len(vc) == len(v):
        kw["vertexcolor"] = (np.clip(vc,0,1)*255).astype(np.uint8)
    else:
        kw["color"] = "lightgray"
    fig = go.Figure(go.Mesh3d(**kw))
    fig.update_layout(title=title, height=540,
                      margin=dict(l=0,r=0,t=30,b=0),
                      scene=dict(aspectmode="data"))
    fig.show()

log(f"device {DEVICE}")

[   0.0s] device cuda


In [ ]:
# A3 – Upload original and empty-room images
# ---------------------------------------------------------------------
banner("A3 – upload images")
from google.colab import files

def load_resized(p):
    im = Image.open(p).convert("RGB")
    w, h = im.size
    s = CFG["MAX_DIM"] / max(w, h)
    if s < 1:
        im = im.resize((round(w*s), round(h*s)), Image.LANCZOS)
    return im

print("Upload the ORIGINAL room photo:")
up = files.upload()
orig_path = "/content/" + list(up.keys())[0]
orig = load_resized(orig_path)
rgb = np.asarray(orig)
H, W = rgb.shape[:2]

print("Upload the EMPTY-ROOM image (same viewpoint, no objects):")
up = files.upload()
empty_path = "/content/" + list(up.keys())[0]
empty = load_resized(empty_path)
empty_rgb = np.asarray(empty)

fig, ax = plt.subplots(1,2,figsize=(13,5))
ax[0].imshow(orig); ax[0].set_title("original")
ax[1].imshow(empty); ax[1].set_title("empty room")
for a in ax: a.axis("off")
plt.show()

In [ ]:
# A4 – MoGe on the original (scene geometry)
# ---------------------------------------------------------------------
banner("A4 – MoGe scene")
# Try to import MoGe model – the package structure may vary
MoGeModel = None
for p in ["moge.model.v2", "moge.model.v1", "moge.model"]:
    try:
        MoGeModel = getattr(__import__(p, fromlist=["MoGeModel"]), "MoGeModel")
        break
    except Exception:
        pass
if MoGeModel is None:
    raise ImportError("MoGeModel not found – check MoGe installation")

moge = MoGeModel.from_pretrained(CFG["MOGE_ID"]).to(DEVICE).eval()

def moge_infer(im_rgb):
    t = torch.tensor(im_rgb / 255.0, dtype=torch.float32, device=DEVICE).permute(2,0,1)
    with torch.no_grad():
        out = moge.infer(t)
    def get(k):
        return out[k].detach().cpu().numpy() if k in out and out[k] is not None else None
    return get("points"), get("mask").astype(bool), get("depth"), get("intrinsics")

points, valid_mask, depth, intr = moge_infer(rgb)
log(f"scene valid points {int(valid_mask.sum()):,}")

# Show depth
d = depth.copy()
d[~valid_mask] = np.nan
plt.figure(figsize=(7,5))
plt.imshow(d, cmap="turbo")
plt.colorbar(fraction=0.046)
plt.title("scene depth"); plt.axis("off")
plt.show()

In [ ]:
# A5 – Scene point cloud
# ---------------------------------------------------------------------
banner("A5 – scene cloud")
P = points[valid_mask]
C = rgb[valid_mask] / 255.0
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(P)
pcd.colors = o3d.utility.Vector3dVector(C)
o3d.io.write_point_cloud(f"{OUT}/scene_cloud.ply", pcd)
show_pcd(P, C, "scene point cloud")

# Store for later use
scene_points = P
scene_colors = C

In [ ]:
# ---------------------------------------------------------------------
# A6 – Grounded-SAM (fixed mask extraction)
# ---------------------------------------------------------------------
banner("A6 – Grounded-SAM")
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, SamModel, SamProcessor

gd_id = "IDEA-Research/grounding-dino-tiny"
gp = AutoProcessor.from_pretrained(gd_id)
gm = AutoModelForZeroShotObjectDetection.from_pretrained(gd_id).to(DEVICE).eval()

sam = SamModel.from_pretrained("facebook/sam-vit-base").to(DEVICE).eval()
sp = SamProcessor.from_pretrained("facebook/sam-vit-base")

prompt = ". ".join(CFG["OBJECTS"]) + " ."
inputs = gp(images=orig, text=prompt, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = gm(**inputs)
results = gp.post_process_grounded_object_detection(
    outputs, inputs["input_ids"],
    threshold=CFG["BOX_THR"],
    text_threshold=CFG["TXT_THR"],
    target_sizes=[orig.size[::-1]]
)[0]

boxes = results["boxes"].cpu().numpy()
labels = results.get("text_labels", results.get("labels", []))
log(f"detected {len(boxes)} objects: {labels}")

obj_masks = []
for b in boxes:
    sam_inputs = sp(orig, input_boxes=[[b.tolist()]], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        sam_out = sam(**sam_inputs)
    # post_process_masks returns a list of masks for each image; each mask may be a list of tensors.
    masks_per_image = sp.image_processor.post_process_masks(
        sam_out.pred_masks.cpu(),
        sam_inputs["original_sizes"].cpu(),
        sam_inputs["reshaped_input_sizes"].cpu()
    )[0]  # first image

    # masks_per_image could be a list of tensors (one per mask) or a single tensor
    if isinstance(masks_per_image, list):
        # take the first mask (usually the best one)
        mask_tensor = masks_per_image[0]
    else:
        mask_tensor = masks_per_image
        # if tensor has shape (num_masks, H, W), take first
        if mask_tensor.dim() == 3 and mask_tensor.shape[0] > 1:
            mask_tensor = mask_tensor[0]

    # Convert to numpy and ensure shape (H, W)
    m_np = mask_tensor.cpu().numpy().astype(bool)
    # If still >2D, squeeze and take first if needed
    if m_np.ndim > 2:
        m_np = m_np.squeeze()
        if m_np.ndim > 2:
            m_np = m_np[0]
    # Reshape to (H,W) – if size mismatches, flatten and reshape
    if m_np.size != H * W:
        m_np = m_np.reshape(H, W)
    obj_masks.append(m_np)

# Visualize as before
overlay = rgb.copy()
for i, m in enumerate(obj_masks):
    col = (np.array(plt.cm.tab10(i % 10)[:3]) * 255).astype(np.uint8)
    overlay[m] = (0.5 * overlay[m] + 0.5 * col).astype(np.uint8)
plt.figure(figsize=(8,6))
plt.imshow(overlay)
plt.title(f"masks: {labels}")
plt.axis("off")
plt.show()

In [ ]:
# A7 – Segmented clouds + clean object images + floor points
# ---------------------------------------------------------------------
banner("A7 – segmented clouds")
all_obj = np.zeros((H,W), dtype=bool)
meta = {
    "objects": [],
    "intrinsics": intr.tolist() if intr is not None else None,
    "width": int(W),
    "height": int(H)
}

cluster_points_list = []
cluster_colors_list = []

for i, (m, lab) in enumerate(zip(obj_masks, labels)):
    mm = m & valid_mask
    if mm.sum() < 50:
        continue
    cp = points[mm]
    cc = rgb[mm] / 255.0
    # save cluster
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(cp)
    pc.colors = o3d.utility.Vector3dVector(cc)
    o3d.io.write_point_cloud(f"{OUT}/clusters/obj_{i:02d}.ply", pc)

    # clean object image on white bg, cropped to bbox
    ys, xs = np.where(m)
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    crop = np.ones((y1-y0+1, x1-x0+1, 3), dtype=np.uint8) * 255
    sub = rgb[y0:y1+1, x0:x1+1]
    sm = m[y0:y1+1, x0:x1+1]
    crop[sm] = sub[sm]
    Image.fromarray(crop).save(f"{OUT}/objects/obj_{i:02d}.png")

    # mask
    Image.fromarray((m.astype(np.uint8) * 255)).save(f"{OUT}/masks/obj_{i:02d}.png")

    # metadata
    meta["objects"].append({
        "id": f"obj_{i:02d}",
        "label": str(lab),
        "n_points": int(mm.sum()),
        "bbox": [int(x0), int(y0), int(x1), int(y1)]  # store for later resync
    })
    all_obj |= m
    cluster_points_list.append(cp)
    cluster_colors_list.append(np.tile(plt.cm.tab10(i % 10)[:3], (len(cp), 1)))
    log(f"obj_{i:02d} {lab}: {int(mm.sum()):,} pts")

# floor points (scene minus all detected objects)
floor_mask = valid_mask & (~all_obj)
fp = points[floor_mask]
o3d.io.write_point_cloud(f"{OUT}/floor.ply",
                         o3d.geometry.PointCloud(o3d.utility.Vector3dVector(fp)))

# save meta
json.dump(meta, open(f"{OUT}/meta.json", "w"), indent=2)

# show segmented clusters
if cluster_points_list:
    show_pcd(np.vstack(cluster_points_list),
             np.vstack(cluster_colors_list),
             "segmented object clusters")

In [ ]:
# A7b – Download raw crops for Gemini editing (optional)
# ---------------------------------------------------------------------
banner("A7b – download crops for Gemini")
shutil.make_archive("/content/object_crops", "zip", f"{OUT}/objects")
try:
    files.download("/content/object_crops.zip")
except Exception:
    pass
print("Edit each obj_XX.png in Gemini (amodal completion / remove shadows).")
print("Keep the same filename (obj_XX.png) and re‑upload in the next cell.")

In [ ]:
# ---------------------------------------------------------------------
# A7c – Upload Gemini‑edited images with aspect‑ratio preservation
# ---------------------------------------------------------------------
banner("A7c – upload Gemini edits (aspect‑ratio aware)")
print("Upload your edited object images (must be square with white background):")
uploaded = files.upload()

def fit_mask_to_crop(src_mask, target_w, target_h):
    """
    Place src_mask (binary) into a canvas of size (target_h, target_w)
    while preserving the aspect ratio of the foreground object.
    The source mask is assumed to be square (e.g., 1024x1024) with padding.
    """
    if src_mask.size == 0:
        return np.zeros((target_h, target_w), dtype=bool)

    # Find the bounding box of the foreground (non‑white area)
    ys, xs = np.where(src_mask)
    if len(xs) == 0:
        return np.zeros((target_h, target_w), dtype=bool)

    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    obj_h = y1 - y0 + 1
    obj_w = x1 - x0 + 1

    # Crop the object tight
    obj_mask = src_mask[y0:y1+1, x0:x1+1]

    # Compute scale so that the object fits inside the target crop,
    # preserving its aspect ratio (fit = min scale to fully contain it)
    scale = min(target_w / obj_w, target_h / obj_h)
    new_w = int(round(obj_w * scale))
    new_h = int(round(obj_h * scale))

    # Resize the object mask to the new size (NEAREST for binary)
    if new_w > 0 and new_h > 0:
        obj_resized = np.array(Image.fromarray(obj_mask).resize((new_w, new_h), Image.NEAREST))
    else:
        return np.zeros((target_h, target_w), dtype=bool)

    # Center the resized object in the target canvas
    canvas = np.zeros((target_h, target_w), dtype=bool)
    y_offset = (target_h - new_h) // 2
    x_offset = (target_w - new_w) // 2
    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = obj_resized
    return canvas

for filename, content in uploaded.items():
    # Load image
    img = Image.open(io.BytesIO(content)).convert("RGB")
    arr = np.array(img)

    # Extract ID
    m_id = re.search(r"obj[_-]?(\d+)", filename)
    if not m_id:
        print(f"Skipping {filename} – no ID")
        continue
    oid = f"obj_{int(m_id.group(1)):02d}"
    print(f"Processing {filename} -> {oid}")

    # Save the edited image (overwrites raw)
    img.save(f"{OUT}/objects/{oid}.png")

    # Find object in meta
    entry = next((o for o in meta["objects"] if o["id"] == oid), None)
    if entry is None:
        print(f"  Warning: {oid} not in meta, skipping sync")
        continue

    # Get original bbox from meta
    x0, y0, x1, y1 = entry["bbox"]
    crop_w = x1 - x0 + 1
    crop_h = y1 - y0 + 1

    # Threshold the edited image to get a binary mask (white background -> 245 threshold)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    _, new_mask_thresh = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY_INV)
    src_mask = new_mask_thresh.astype(bool)

    # Fit the source mask into the original crop while preserving aspect ratio
    fitted_mask = fit_mask_to_crop(src_mask, crop_w, crop_h)

    # Place into the full image frame
    full_mask = np.zeros((H, W), dtype=bool)
    full_mask[y0:y1+1, x0:x1+1] = fitted_mask
    # Intersect with MoGe valid depth mask
    full_mask = full_mask & valid_mask

    # Save updated mask
    Image.fromarray((full_mask.astype(np.uint8) * 255)).save(f"{OUT}/masks/{oid}.png")

    # Update cluster point cloud
    cluster_pts = points[full_mask]
    cluster_cols = rgb[full_mask] / 255.0

    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(cluster_pts)
    pc.colors = o3d.utility.Vector3dVector(cluster_cols)
    o3d.io.write_point_cloud(f"{OUT}/clusters/{oid}.ply", pc)

    # Update meta with new point count
    entry["n_points"] = int(full_mask.sum())
    print(f"  ✅ {oid}: updated with {full_mask.sum():,} 3D points (aspect‑ratio preserved)")

# Save updated meta
json.dump(meta, open(f"{OUT}/meta.json", "w"), indent=2)
print("All Gemini images processed. Clusters and masks are now synchronized.")

In [11]:
# A8 – Background shell from empty-room image
# ---------------------------------------------------------------------
banner("A8 – background mesh")
ep, em, ed, _ = moge_infer(empty_rgb)
Hb, Wb = ed.shape

# Build quad mesh from depth
idx = np.full((Hb, Wb), -1, dtype=np.int64)
idx[em] = np.arange(int(em.sum()))
V = ep[em]
C_bg = empty_rgb[em] / 255.0

tl, tr, bl, br = idx[:-1,:-1], idx[:-1,1:], idx[1:,:-1], idx[1:,1:]
dq = [ed[:-1,:-1], ed[:-1,1:], ed[1:,:-1], ed[1:,1:]]
keep = (tl >= 0) & (tr >= 0) & (bl >= 0) & (br >= 0)
keep &= (np.maximum.reduce(dq) - np.minimum.reduce(dq)) < CFG["REL_JUMP"] * (sum(dq)/4)

F1 = np.stack([tl[keep], bl[keep], tr[keep]], axis=1)
F2 = np.stack([tr[keep], bl[keep], br[keep]], axis=1)
F = np.concatenate([F1, F2])

bg_mesh = o3d.geometry.TriangleMesh()
bg_mesh.vertices = o3d.utility.Vector3dVector(V)
bg_mesh.vertex_colors = o3d.utility.Vector3dVector(np.clip(C_bg,0,1))
bg_mesh.triangles = o3d.utility.Vector3iVector(F)
bg_mesh.remove_unreferenced_vertices()
bg_mesh.remove_degenerate_triangles()
bg_mesh.compute_vertex_normals()
o3d.io.write_triangle_mesh(f"{OUT}/background_room.glb", bg_mesh)
log(f"background {len(bg_mesh.vertices):,}v {len(bg_mesh.triangles):,}f")
show_mesh(bg_mesh, "background room shell")


  A8 – background mesh
[ 616.9s] background 1,026,048v 2,048,046f


In [ ]:
# A9 – Package outputs for download
# ---------------------------------------------------------------------
banner("A9 – package")
shutil.make_archive("/content/notebookA_outputs", "zip", OUT)
print("contents of", OUT)
for p in sorted(os.listdir(OUT)):
    print("  ", p)
try:
    files.download("/content/notebookA_outputs.zip")
except Exception:
    print("Download /content/notebookA_outputs.zip from Files panel")

print("\n✅ Notebook A complete. Outputs are ready for Notebook B and C.")